# Notebook 5. Cấp 7 (PUCT + mạng policy, value) và RQ3

Người phụ trách: Nguyễn Trọng Phúc.

Nội dung:
1. Huấn luyện mạng value trên các thế cờ của ván chuyên gia (nb3), nhãn là điểm Alpha-Beta tìm sâu.
2. Chọn hệ số khám phá `c_puct` bằng ván thử riêng (phiên bản 1 đã chọn `mix = 0` theo cùng cách).
3. Ablation cấp 7: MCTS thuần, thêm prior, thêm value, đầy đủ (cùng một hàm, khác hai công tắc).
4. RQ3: cấp 7 so với cấp 4 ở cùng số mô phỏng và cùng thời gian; cấp 7 so với cấp 5.

Thành phần AlphaGo đã hiện thực: mạng policy cho tiên nghiệm, mạng policy nhanh cho mô phỏng, mạng value, tìm kiếm cây PUCT, trộn value và mô phỏng theo `mix`.
Chưa hiện thực: mạng rollout tuyến tính riêng, tìm kiếm song song, mạng value học từ ván tự chơi của mạng RL quy mô lớn.

Cần có: `files/games_*.p`, `files/nhan_alphabeta.p` và `files/policy_conn.keras` (nb3).
**Lưu ý:** mục 4b chạy theo thời gian thật, lúc đó không chạy việc nặng khác trên máy.

In [ ]:
# QUICK = True: chạy thử vài ván, vài epoch để kiểm tra code (ghi vào thư mục con thu_nhanh/)
# QUICK = False: chạy thật theo config.yaml, kết quả dùng cho báo cáo
QUICK = False

import os
if not os.path.exists("config.yaml"):
    raise RuntimeError("Hãy mở Jupyter trong thư mục final_connect_four (nơi có config.yaml).")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.experiments import setup, summaries_table, with_random_opening, move_times_of
from utils.arena import run_match, format_summary, read_log, first_player_stats
from utils.data import machine_info
import keras
from utils import data, nets, tactics
from utils.experiments import plot_scores, info_mean, records_of
ex = setup(QUICK)
cfg = ex.cfg
print("Cấu hình máy:", machine_info())

## 1. Mạng value

Mạng value cho biết thế cờ tốt hay xấu với người sắp đi, trong [−1, 1]; cấp 7 dùng nó để chấm lá của cây tìm kiếm.

Lần đầu của phiên bản 2 học kết quả ván (3 lớp hòa / thắng / thua, như sách chương 15): val_loss thấp nhất ở epoch 2 rồi tăng, test acc 0,69, và cấp 7 với mạng đó thua cấp 4 ở mọi `c_puct` thử. Lý do: cùng một thế cờ, kết quả ván còn phụ thuộc mọi nước sai về sau, nên nhãn rất nhiễu và mạng học thuộc ván. Kết quả lần đó được giữ trong `results/v2_lan1/`.

Cách hiện tại (`train_value.target: search_score`), mạng học **điểm** của Alpha-Beta tìm sâu 5 (hồi quy, đầu ra tanh, loss MSE) trên hai nguồn thế cờ:
- 66.314 thế của ván chuyên gia, đã gán nhãn ở nb3;
- thế cờ mới từ `self_play_games` ván tự chơi của mạng cấp 5 (rút nước theo xác suất của mạng, mỗi nước `self_play_random_prob` khả năng đi ngẫu nhiên), gán nhãn cùng cách. Giống AlphaGo lấy dữ liệu cho mạng value từ ván tự chơi có chèn nước ngẫu nhiên: ván chuyên gia ít khi có thế sau một nước sai, trong khi cây tìm kiếm của cấp 7 phải chấm cả những thế đó.

Nhãn của mỗi thế, theo góc người sắp đi:
- thế tìm kiếm chứng minh người sắp đi thắng chắc: +1; thua chắc: −1;
- thế còn lại: tanh(k × điểm lượng giá). Điểm lượng giá có thang tùy ý (chia 276 chỉ để nhỏ hơn điểm thắng) và phần lớn chỉ trong ±0,07, nên được đổi sang thang kết quả ván. k chọn trong {1, 2, 5, 10, 20, 50, 100} sao cho tanh(k × điểm) gần kết quả ván nhất (sai số bình phương), chỉ trên các ván của tập train.

Chia train/val/test theo ván như nb3; tập train thêm bản lật gương; dừng sớm theo val_loss và giữ trọng số tốt nhất.

In [ ]:
paths = []
for part in cfg["data"]["seeds"]:
    path = ex.model(f"games_{part}.p")
    if os.path.exists(path):
        paths.append(path)
    else:
        print("CẢNH BÁO: thiếu", path)
games = data.merge_game_files(paths)
print("Số ván:", len(games))
labeled_expert = data.load_games(ex.model("nhan_alphabeta.p"))
print("Số thế của ván chuyên gia đã gán nhãn (nb3):", len(labeled_expert))

In [ ]:
# ván tự chơi của mạng cấp 5: sinh một lần rồi lưu, chạy lại thì đọc file
value_cfg = cfg["train_value"]
self_play_path = ex.model("tu_choi_cap5.p")
if os.path.exists(self_play_path):
    self_games = data.load_games(self_play_path)
    print("Đọc", len(self_games), "ván tự chơi từ", self_play_path)
else:
    policy_model = nets.load_model(ex.model("policy_conn.keras"))
    start = time.time()
    self_games = data.self_play_games(policy_model, ex.n(value_cfg["self_play_games"], 10),
                                      random_prob=value_cfg["self_play_random_prob"],
                                      seed=value_cfg["self_play_seed"])
    seconds = round(time.time() - start, 1)
    data.save_games(self_games, self_play_path,
                    {"n_games": len(self_games), "random_prob": value_cfg["self_play_random_prob"],
                     "seed": value_cfg["self_play_seed"], "seconds": seconds, "machine": machine_info()})
    print("Sinh", len(self_games), "ván tự chơi trong", seconds, "giây")
lengths = [len(g["history"]) for g in self_games]
print(f"Số nước trung bình mỗi ván tự chơi: {np.mean(lengths):.1f}")

# chỉ gán nhãn các thế chưa có trong dữ liệu của nb3; chạy song song, bị ngắt thì chạy lại sẽ làm tiếp
positions = data.new_positions(self_games, labeled_expert)
print("Thế cờ mới cần gán nhãn:", len(positions))
labeled_self = data.label_positions(positions, depth=cfg["data"]["label_depth"],
                                    processes=cfg["data"]["label_processes"],
                                    out_path=ex.model("nhan_alphabeta_tu_choi.p"))
labeled = labeled_expert + labeled_self
print("Tổng số thế đã gán nhãn dùng cho mạng value:", len(labeled))

In [ ]:
best_scores = np.array([item[3] for item in labeled])
print(f"Thắng chắc: {np.mean(best_scores >= 1):.3f}, thua chắc: {np.mean(best_scores <= -1):.3f}, "
      f"còn lại: {np.mean(np.abs(best_scores) < 1):.3f}")

score_scale, scale_errors = data.fit_score_scale(games, labeled, seed=0)
scale_table = pd.DataFrame({"k": list(scale_errors.keys()), "sai số bình phương với kết quả ván (train)": list(scale_errors.values())}).round(4)
scale_table.to_csv(ex.table("nb5_chon_he_so_k.csv"), index=False, encoding="utf-8-sig")
print("Chọn k =", score_scale)
display(scale_table)

value_set = data.make_dataset(games, kind="search_score", seed=0, mirror=cfg["train_value"]["mirror"],
                              labeled=labeled, score_scale=score_scale)
print(data.describe_dataset(value_set))
mean_train = float(np.mean(value_set["y_train"]))
baseline_mse = float(np.mean((value_set["y_test"] - mean_train) ** 2))
baseline_mae = float(np.mean(np.abs(value_set["y_test"] - mean_train)))
print(f"Mốc 'luôn đoán trung bình của train' trên test: MSE {baseline_mse:.3f}, MAE {baseline_mae:.3f}")

In [ ]:
keras.utils.set_random_seed(0)
value_model, value_report = nets.train_value(games, path=ex.model("value_conn.keras"),
                                             epochs=ex.n(value_cfg["epochs"], 2),
                                             batch_size=value_cfg["batch_size"],
                                             learning_rate=value_cfg["learning_rate"], seed=0, verbose=2,
                                             mirror=value_cfg["mirror"], patience=value_cfg["patience"],
                                             labeled=labeled, target=value_cfg["target"])
print(f"Chạy {value_report['epochs_run']} epoch, giữ trọng số epoch {value_report['best_epoch']} (val_loss thấp nhất)")
print(f"Huấn luyện {value_report['train_seconds']} giây")
print(f"val: MSE {value_report['val_loss']:.3f}, MAE {value_report['val_mae']:.3f}")
print(f"test: MSE {value_report['test_loss']:.3f}, MAE {value_report['test_mae']:.3f} "
      f"(mốc: MSE {baseline_mse:.3f}, MAE {baseline_mae:.3f})")
print("Số tham số của mạng value:", value_model.count_params())

In [ ]:
curve = value_report["curve"]
epochs = range(1, len(curve["loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, curve["loss"], label="train")
axes[0].plot(epochs, curve["val_loss"], label="val")
axes[0].axhline(baseline_mse, color="#999999", linestyle="--", label="luôn đoán trung bình")
axes[0].set_title("Loss (MSE)")
axes[1].plot(epochs, curve["mae"], label="train")
axes[1].plot(epochs, curve["val_mae"], label="val")
axes[1].axhline(baseline_mae, color="#999999", linestyle="--", label="luôn đoán trung bình")
axes[1].set_title("Sai số tuyệt đối trung bình (MAE)")
for ax in axes[:2]:
    ax.axvline(value_report["best_epoch"], color="#999999", linestyle=":", linewidth=1)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.25)
    ax.legend()

# dự đoán so với nhãn trên tập test (mỗi chấm một thế cờ)
predicted = np.squeeze(value_model.predict(value_set["X_test"], verbose=0))
axes[2].scatter(value_set["y_test"], predicted, s=3, alpha=0.2)
axes[2].plot([-1, 1], [-1, 1], color="#999999", linestyle="--", linewidth=1)
axes[2].set_xlabel("nhãn (điểm tìm kiếm đã đổi thang)")
axes[2].set_ylabel("dự đoán của mạng")
axes[2].set_title("Tập test: dự đoán so với nhãn")
axes[2].grid(alpha=0.25)
fig.suptitle(f"Mạng value, giữ epoch {value_report['best_epoch']}")
fig.tight_layout()
fig.savefig(ex.fig("nb5_duong_cong_value.png"), dpi=150)

# thế thắng chắc / thua chắc trên test: mạng có nhận ra đúng dấu không
proven = np.abs(value_set["y_test"]) >= 1.0
same_sign = np.mean(np.sign(predicted[proven]) == np.sign(value_set["y_test"][proven]))
print(f"Thế thắng chắc / thua chắc trên test: {int(proven.sum())} thế, mạng đoán đúng dấu {same_sign:.3f}")

## 2. Chọn c_puct bằng ván thử riêng

`c_puct` quyết định cấp 7 tin vào prior của mạng policy đến đâu: nhỏ thì dồn mô phỏng vào nước mạng thích, lớn thì thử rộng hơn.
Mỗi giá trị đấu cùng một đối thủ (`cap7_pilot.opponent`, cấp 4 mặc định), seed riêng `cap7_pilot.seed`, không dùng ván của RQ3 hay giải đấu cuối.
Chọn giá trị có score cao nhất; các mục sau của notebook này dùng giá trị đó. `mix` giữ như `config.yaml` (0.0, đã chọn ở phiên bản 1).
Dải thử: lần đầu {1,0; 1,5; 3,0} với mạng value cũ cho 1,0 tốt nhất, nằm ở mép dải, nên dải hiện tại mở rộng xuống 0,5 và 0,7 (xem `config.yaml`).

Phiên bản 2 bật `win_check`: ở mỗi lá, người sắp đi có nước thắng ngay thì giá trị lá là +1 chắc chắn thay cho giá trị mạng value. Sau khi chọn `c_puct`, cùng các ván đó (cùng seed, cùng khai cuộc) được chơi lại với `win_check` tắt để đo riêng tác dụng của nó.

**Trước khi chạy nb6:** ghi giá trị đã chọn vào `agents.cap7.params.c_puct` trong `config.yaml`.

In [ ]:
pilot_cfg = cfg["cap7_pilot"]
n_games = ex.n(pilot_cfg["games"], 4)
opening = pilot_cfg["opening_random_moves"]
log_path = ex.log("nb5_chon_c_puct.jsonl")
pilot_opponent = with_random_opening(ex.agent(pilot_cfg["opponent"]), opening)
chosen_mix = cfg["agents"]["cap7"]["params"]["mix"]

pilot_summaries = []
for k, c in enumerate(pilot_cfg["c_puct"]):
    agent = with_random_opening(ex.agent("cap7", overrides={"c_puct": c}), opening)
    s = run_match(agent, pilot_opponent, f"cap7_c{c}", pilot_cfg["opponent"], n_games,
                  seed=pilot_cfg["seed"] + k, log_path=log_path)
    pilot_summaries.append(s)
    print(format_summary(s))

best = 0
for k in range(len(pilot_summaries)):
    if pilot_summaries[k]["score"] > pilot_summaries[best]["score"]:
        best = k
chosen_c = pilot_cfg["c_puct"][best]

# cùng seed với lần chạy của c_puct đã chọn, chỉ tắt win_check
agent = with_random_opening(ex.agent("cap7", overrides={"c_puct": chosen_c, "win_check": False}), opening)
s = run_match(agent, pilot_opponent, f"cap7_c{chosen_c}_tat_win_check", pilot_cfg["opponent"], n_games,
              seed=pilot_cfg["seed"] + best, log_path=log_path)
print(format_summary(s))
pilot_summaries.append(s)

pilot_table = summaries_table(pilot_summaries)
pilot_table.insert(1, "c_puct", list(pilot_cfg["c_puct"]) + [chosen_c])
pilot_table.insert(2, "win_check", [True] * len(pilot_cfg["c_puct"]) + [False])
pilot_table.to_csv(ex.table("nb5_chon_c_puct.csv"), index=False, encoding="utf-8-sig")
print(f"Kết luận: chọn c_puct = {chosen_c}, mix = {chosen_mix}")
if cfg["agents"]["cap7"]["params"]["c_puct"] != chosen_c:
    print(f"NHỚ: sửa agents.cap7.params.c_puct thành {chosen_c} trong config.yaml trước khi chạy nb6.")
pilot_table

## 3. Ablation cấp 7

Bốn cấu hình cùng 200 mô phỏng, cùng đấu một đối thủ chung (`rq3.opponent`, cấp 3). Mỗi ván mở đầu bằng 2 nước ngẫu nhiên có seed.

In [ ]:
rq3_cfg = cfg["rq3"]
n_games = ex.n(rq3_cfg["games_per_pair"], 4)
opening = rq3_cfg["opening_random_moves"]
log_path = ex.log("nb5_ablation_cap7.jsonl")
opponent_name = rq3_cfg["opponent"]
opponent = with_random_opening(ex.agent(opponent_name), opening)

ablation_summaries = []
for k, name in enumerate(["mcts_thuan", "them_prior", "them_value", "day_du"]):
    # cùng c_puct đã chọn cho cả bốn cấu hình, chỉ khác hai công tắc prior và value
    agent = with_random_opening(ex.agent(name, overrides={"c_puct": chosen_c}, section="rq3_ablation"), opening)
    s = run_match(agent, opponent, name, opponent_name, n_games, seed=rq3_cfg["seed"] + k, log_path=log_path)
    ablation_summaries.append(s)
    print(format_summary(s))

ablation_table = summaries_table(ablation_summaries)
ablation_table.to_csv(ex.table("nb5_ablation_cap7.csv"), index=False, encoding="utf-8-sig")
ablation_table

## 4. RQ3

### 4a. Cùng số mô phỏng mỗi nước

In [ ]:
sim_list = rq3_cfg["simulations"] if not QUICK else [20, 50]
log_path = ex.log("nb5_rq3_mo_phong.jsonl")
sim_summaries = []
for k, sims in enumerate(sim_list):
    name7 = f"cap7_{sims}mp"
    name4 = f"cap4_{sims}mp"
    a = ex.agent("cap7", overrides={"num_simulations": sims, "mix": chosen_mix, "c_puct": chosen_c})
    b = ex.agent("cap4", overrides={"num_simulations": sims})
    start = time.perf_counter()
    s = run_match(a, b, name7, name4, n_games, seed=rq3_cfg["seed"] + 10 + k, log_path=log_path)
    sim_summaries.append(s)
    print(format_summary(s), "| chạy mất", round(time.perf_counter() - start), "giây")

sim_table = summaries_table(sim_summaries)
sim_table.insert(1, "mô phỏng/nước", sim_list)
sim_table.to_csv(ex.table("nb5_rq3_mo_phong.csv"), index=False, encoding="utf-8-sig")
sim_table

### 4b. Cùng thời gian mỗi nước

Cấp 7 gọi mạng ở mỗi lá nên mỗi mô phỏng tốn hơn cấp 4 nhiều lần; cùng thời gian thì cấp 7 làm được ít mô phỏng hơn. Bảng ghi số mô phỏng trung bình của mỗi bên.

In [ ]:
time_limits = rq3_cfg["time_limits"] if not QUICK else [0.05, 0.1]
log_path = ex.log("nb5_rq3_thoi_gian.jsonl")
time_summaries = []
extra = []
for k, t in enumerate(time_limits):
    name7 = f"cap7_{t}s"
    name4 = f"cap4_{t}s"
    a = ex.agent("cap7", overrides={"time_limit": t, "mix": chosen_mix, "c_puct": chosen_c})
    b = ex.agent("cap4", overrides={"time_limit": t})
    start = time.perf_counter()
    s = run_match(a, b, name7, name4, n_games, seed=rq3_cfg["seed"] + 20 + k, log_path=log_path)
    time_summaries.append(s)
    records = records_of(log_path, name7, name4)
    extra.append({"giây/nước": t,
                  "mô phỏng TB cấp 7": info_mean(records, name7, "simulations"),
                  "mô phỏng TB cấp 4": info_mean(records, name4, "simulations")})
    print(format_summary(s), "| chạy mất", round(time.perf_counter() - start), "giây")

time_table = pd.concat([summaries_table(time_summaries), pd.DataFrame(extra)], axis=1)
time_table.to_csv(ex.table("nb5_rq3_thoi_gian.csv"), index=False, encoding="utf-8-sig")
time_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, x_values, series, xlabel in [(axes[0], sim_list, sim_summaries, "số mô phỏng mỗi nước"),
                                     (axes[1], time_limits, time_summaries, "thời gian mỗi nước (giây)")]:
    scores = [s["score"] for s in series]
    low = [max(0.0, s["score"] - s["ci_low"]) for s in series]
    high = [max(0.0, s["ci_high"] - s["score"]) for s in series]
    ax.errorbar(x_values, scores, yerr=[low, high], marker="o", capsize=4, linewidth=2)
    ax.axhline(0.5, color="#999999", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("score của cấp 7 khi đấu cấp 4")
    ax.grid(alpha=0.25)
axes[0].set_title("Cùng số mô phỏng")
axes[1].set_title("Cùng thời gian")
fig.suptitle("RQ3: cấp 7 đấu cấp 4")
fig.tight_layout()
fig.savefig(ex.fig("nb5_rq3.png"), dpi=150)

### 4c. Cấp 7 so với cấp 5 và độ chính xác chiến thuật

Cấu hình mặc định của mỗi cấp (theo `config.yaml`).

In [ ]:
log_path = ex.log("nb5_rq3_cap5.jsonl")
cap7 = with_random_opening(ex.agent("cap7", overrides={"mix": chosen_mix, "c_puct": chosen_c}), opening)
cap5 = with_random_opening(ex.agent("cap5"), opening)
s = run_match(cap7, cap5, "cap7", "cap5", n_games, seed=rq3_cfg["seed"] + 30, log_path=log_path)
print(format_summary(s))
summaries_table([s]).to_csv(ex.table("nb5_rq3_cap7_cap5.csv"), index=False, encoding="utf-8-sig")

In [ ]:
tactics_path = ex.model("tactics.json")
if os.path.exists(tactics_path):
    positions = tactics.load_tactics(tactics_path)
    rows = []
    for name in ["cap4", "cap5", "cap7"]:
        if name == "cap7":
            move_fn, info_fn = ex.agent(name, overrides={"mix": chosen_mix, "c_puct": chosen_c})
        else:
            move_fn, info_fn = ex.agent(name)
        row = {"tác tử": name}
        row.update(tactics.score_agent(move_fn, positions))
        rows.append(row)
    for name in ["mcts_thuan", "them_prior", "them_value", "day_du"]:
        move_fn, info_fn = ex.agent(name, overrides={"c_puct": chosen_c}, section="rq3_ablation")
        row = {"tác tử": name}
        row.update(tactics.score_agent(move_fn, positions))
        rows.append(row)
    rq3_tactics = pd.DataFrame(rows).round(3)
    rq3_tactics.to_csv(ex.table("nb5_chien_thuat_cap7.csv"), index=False, encoding="utf-8-sig")
    display(rq3_tactics)
else:
    print("Chưa có", tactics_path, "- chạy mục 1 của nb2 (hoặc chép file từ máy kia) rồi chạy lại ô này.")